# PhysicsNeMo Darcy FNO Colab Starter

이 노트북은 NVIDIA PhysicsNeMo의 `examples/cfd/darcy_fno` 예제를 Colab에서 먼저 돌려보고, 베이스 FNO 모델 위에 커스텀 아이디어를 얹기 위한 출발점입니다.

흐름:
1. Colab GPU/CUDA 확인
2. PhysicsNeMo 설치
3. 공식 repo clone
4. Darcy FNO smoke test용 config 축소
5. 베이스라인 학습 실행
6. 커스텀 실험 방향 잡기

공식 예제: https://github.com/NVIDIA/physicsnemo/tree/main/examples/cfd/darcy_fno

In [ ]:
# 0. Runtime auto-fix: GPU, CUDA, UTF-8 encoding
import os
import sys
import locale
import subprocess
import torch

# Colab/Python UTF-8 guard. This fixes common locale/encoding errors from pip or subprocesses.
os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["PYTHONUTF8"] = "1"
os.environ.setdefault("LANG", "C.UTF-8")
os.environ.setdefault("LC_ALL", "C.UTF-8")
try:
    locale.getpreferredencoding = lambda do_setlocale=True: "UTF-8"
except Exception as exc:
    print("UTF-8 locale patch skipped:", repr(exc))

print("python:", sys.version.split()[0])
print("preferred encoding:", locale.getpreferredencoding())

try:
    subprocess.run(["nvidia-smi"], check=False)
except FileNotFoundError:
    print("nvidia-smi not found. Continuing with CPU fallback.")

HAS_GPU = torch.cuda.is_available()
DEVICE = "cuda" if HAS_GPU else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if HAS_GPU else "CPU fallback"

print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("device:", DEVICE)
print("gpu:", GPU_NAME)

AUTO_FIX = {
    "has_gpu": HAS_GPU,
    "device": DEVICE,
    "gpu_name": GPU_NAME,
    "encoding": locale.getpreferredencoding(),
}

if not HAS_GPU:
    print("WARNING: GPU is not available. The notebook will use a tiny CPU smoke-test config or skip heavy training if needed.")

## 1. Drive 마운트
체크포인트와 실험 로그를 세션 종료 후에도 남기려면 Drive를 마운트합니다.

In [ ]:
RUN_ROOT = "/content/physicsnemo_runs/darcy_fno"

try:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_ROOT = "/content/drive/MyDrive/physicsnemo_runs/darcy_fno"
except Exception as exc:
    print("Drive mount skipped:", repr(exc))
    print("Using temporary runtime storage instead.")

!mkdir -p "$RUN_ROOT"
print(RUN_ROOT)

## 2. 설치
Darcy FNO smoke test에는 먼저 최소 설치를 사용합니다. `nn-extras`는 무겁고 Colab 런타임 재시작을 유발할 수 있어서, 필요한 실험에서만 나중에 추가하세요.

In [ ]:
!pip install -U pip
!pip install nvidia-physicsnemo hydra-core termcolor warp-lang

In [ ]:
# 설치 확인
import physicsnemo
print("physicsnemo:", physicsnemo.__version__)

## 3. 공식 repo clone
예제 파일은 pip 패키지보다 GitHub repo에서 보는 편이 편합니다.

In [ ]:
%cd /content
![ -d physicsnemo ] || git clone https://github.com/NVIDIA/physicsnemo.git
%cd /content/physicsnemo/examples/cfd/darcy_fno
!ls -la

## 4. 원본 config 확인
공식 Darcy FNO 예제는 다음 요소가 핵심입니다.

- `Darcy2D`: permeability와 darcy solution 샘플을 on-the-fly 생성
- `FNO`: grid field operator learning 모델
- `config.yaml`: 모델 크기, resolution, batch size, pseudo epoch 수 제어

In [ ]:
!sed -n '1,220p' config.yaml

## 5. Colab smoke test config 만들기
원본은 `resolution=256`, `batch_size=64`, `max_pseudo_epochs=256`라 Colab 첫 실행에는 큽니다. 여기서는 GPU가 있으면 작은 config를 만들고, GPU가 없으면 자동으로 더 작은 CPU fallback config를 만듭니다.

In [ ]:
# 5. Colab smoke test config 만들기
# GPU가 있으면 작은 FNO smoke test, GPU가 없으면 더 작은 CPU fallback config를 자동 생성합니다.
from pathlib import Path

if "HAS_GPU" not in globals():
    HAS_GPU = torch.cuda.is_available()

smoke_resolution = 64 if HAS_GPU else 32
smoke_batch_size = 4 if HAS_GPU else 1
smoke_latent_channels = 16 if HAS_GPU else 8
smoke_fno_modes = 8 if HAS_GPU else 4
smoke_epochs = 4 if HAS_GPU else 1
smoke_samples = 16 if HAS_GPU else 2
smoke_valid = 8 if HAS_GPU else 2

smoke_config = f"""arch:
  decoder:
    out_features: 1
    layers: 1
    layer_size: 32
  fno:
    in_channels: 1
    dimension: 2
    latent_channels: {smoke_latent_channels}
    fno_layers: 4
    fno_modes: {smoke_fno_modes}
    padding: 9
normaliser:
  permeability:
    mean: 1.25
    std_dev: .75
  darcy:
    mean: 4.52E-2
    std_dev: 2.79E-2
scheduler:
  initial_lr: 1.E-3
  decay_rate: .85
  decay_pseudo_epochs: 4
training:
  resolution: {smoke_resolution}
  batch_size: {smoke_batch_size}
  rec_results_freq : 2
  max_pseudo_epochs: {smoke_epochs}
  pseudo_epoch_sample_size: {smoke_samples}
validation:
  sample_size: {smoke_valid}
  validation_pseudo_epochs: 1
"""

Path("config_colab_smoke.yaml").write_text(smoke_config, encoding="utf-8")
print(smoke_config)

## 6. smoke test 실행
Hydra config 이름을 바꾸기보다, smoke config를 `config.yaml`로 복사해서 실행합니다. 원본은 백업해둡니다.

In [ ]:
# 6. smoke test 실행
# config를 교체하고 학습을 실행합니다. GPU가 없거나 런타임이 불안정하면 실패 내용을 출력하고 노트북은 계속 진행합니다.
from pathlib import Path
import shutil
import subprocess
import sys

if Path("config.yaml").exists() and not Path("config_original.yaml").exists():
    shutil.copyfile("config.yaml", "config_original.yaml")

shutil.copyfile("config_colab_smoke.yaml", "config.yaml")
print("Using smoke config on", DEVICE if "DEVICE" in globals() else "unknown device")

result = subprocess.run([sys.executable, "train_fno_darcy.py"])

if result.returncode != 0:
    msg = f"train_fno_darcy.py exited with code {result.returncode}."
    if globals().get("HAS_GPU", False):
        raise RuntimeError(msg)
    print(msg)
    print("CPU fallback mode: training may be unsupported or too slow in this runtime. Continue with code reading/customization cells.")

## 7. 실험 결과 보존
공식 예제는 현재 폴더의 `checkpoints/`, 로그/이미지를 생성합니다. Drive로 복사합니다.

In [ ]:
!mkdir -p "$RUN_ROOT/smoke_test"
!cp -r checkpoints "$RUN_ROOT/smoke_test/" 2>/dev/null || true
!find . -maxdepth 3 -type f | sed -n '1,80p'

## 8. 커스텀 아이디어를 넣는 위치

가장 쉬운 순서:

1. config만 바꾸기: `resolution`, `latent_channels`, `fno_modes`, `lr`, `batch_size`
2. loss 바꾸기: MSE + smoothness/gradient penalty
3. 입력 채널 추가: permeability 외에 좌표 grid, boundary mask, physics feature
4. 모델 교체: FNO wrapper를 subclass하거나 decoder/latent size 변경
5. validation metric 추가: relative L2, spectrum error, residual-like metric

아래는 loss를 커스텀하는 가장 작은 예시입니다. 실제 적용은 `train_fno_darcy.py`의 `loss_fun`와 `forward_train` 근처를 복사해서 실험 파일로 분기하는 것을 추천합니다.

In [ ]:
import torch
from torch.nn import MSELoss

mse = MSELoss(reduction="mean")

def gradient_penalty_2d(pred):
    # pred: [B, C, H, W]
    dx = pred[..., 1:, :] - pred[..., :-1, :]
    dy = pred[..., :, 1:] - pred[..., :, :-1]
    return dx.square().mean() + dy.square().mean()

def custom_loss(pred, target, smooth_weight=1e-4):
    return mse(pred, target) + smooth_weight * gradient_penalty_2d(pred)

# 사용 아이디어:
# loss = custom_loss(pred, target, smooth_weight=cfg.custom.smooth_weight)

## 9. 추천 연구 질문 템플릿

Darcy FNO에서 바로 실험하기 좋은 질문들:

- FNO modes 수를 줄이면/늘리면 고주파 error가 어떻게 바뀌나?
- permeability field의 분포가 바뀌면 generalization이 무너지나?
- MSE 대신 relative L2 또는 gradient penalty를 섞으면 boundary/edge가 좋아지나?
- 작은 resolution에서 학습하고 큰 resolution으로 inference하면 얼마나 유지되나?
- 좌표 channel을 추가하면 성능이 좋아지나?

처음 3개 실험만 해도 연구 방향이 꽤 보입니다.

In [ ]:
# 다음 실험용 config 예시: 조금 더 큰 모델
%%writefile config_colab_medium.yaml
arch:
  decoder:
    out_features: 1
    layers: 1
    layer_size: 32
  fno:
    in_channels: 1
    dimension: 2
    latent_channels: 32
    fno_layers: 4
    fno_modes: 12
    padding: 9
normaliser:
  permeability:
    mean: 1.25
    std_dev: .75
  darcy:
    mean: 4.52E-2
    std_dev: 2.79E-2
scheduler:
  initial_lr: 1.E-3
  decay_rate: .85
  decay_pseudo_epochs: 8
training:
  resolution: 128
  batch_size: 8
  rec_results_freq : 4
  max_pseudo_epochs: 32
  pseudo_epoch_sample_size: 256
validation:
  sample_size: 64
  validation_pseudo_epochs: 4

## 10. 원본 복구
공식 예제로 돌아가고 싶으면 아래 셀을 실행하세요.

In [ ]:
!cp config_original.yaml config.yaml